In [85]:
from src.nn.nn_dataset_pre import Datapreprocessor
from omegaconf import OmegaConf
%load_ext autoreload
%autoreload 2
cfg = OmegaConf.load("src/conf/setup_dataset_nn_o_e.yaml")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [92]:

datapreprocessor = Datapreprocessor(cfg)
datapreprocessor.get_preprocess_save_data()
init_condition_table = datapreprocessor.create_save_col_data()
datapreprocessor.update_info_file()




Number of files in the train folder:  8 where  3 are col files and 5 are data files
Number of files in the val folder:  2
Number of files in the test folder:  1
Loading raw data from:  ./data_o_e/SM4_P/dataset_v1/raw
Data saved to:  ./data_o_e/SM4_P/dataset_v1/train\train_data1.h5
Data saved to:  ./data_o_e/SM4_P/dataset_v1/val\val_data1.h5
Data saved to:  ./data_o_e/SM4_P/dataset_v1/test\test_data1.h5
Generating collocation points
Number of different initial conditions for collocation points:  24
['theta', 'omega', 'E_d_dash', 'E_q_dash', 'P_m', 'Vs', 'theta_vs'] Variables
[[-2, 2], [-1, 1], [0], [0.9, 1.1], [0.7], [0.95, 1.05], [-0.3, 0.3]] Set of values for init conditions
[2, 2, 1, 2, 1, 3, 1] Iterations per value
Data saved to:  ./data_o_e/SM4_P/dataset_v1/train\train_data_col4.h5
Data saved to:  ./data_o_e/SM4_P/dataset_v1/train\train_data_init4.h5
Updated info.txt successfully.


In [10]:
#print the shape of the dataset train_loader
from src.nn.nn_dataset_pre import HDF5Dataset, HDF5Dataset_static, HDF5DatasetStatic
from torch.utils.data import DataLoader
import time
from tqdm import tqdm

# clean gc
import gc
gc.collect()


def memory_usage():
    """Returns the current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB


print("Memory usage before loading data:", memory_usage(), "MB")
cfg = OmegaConf.load("src/conf/setup_dataset_nn_o_e.yaml")

number_of_dataset_folder = cfg.dataset.number # Define the number of the dataset to load
folder_path = "./"+cfg.dirs.dataset_dir+"/" + cfg.model.model_flag + '/dataset_v' + str(cfg.dataset.number)
train_dataset = HDF5DatasetStatic(folder_path,'train', step=1, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=30000, shuffle=True)

col_dataset = HDF5DatasetStatic(folder_path,'train', data_name="col", step=1, shuffle=True)
col_loader = DataLoader(col_dataset, batch_size=5, shuffle=True)

init_dataset = HDF5DatasetStatic(folder_path,'train', data_name="data_init", step=1, shuffle=True)
init_loader = DataLoader(init_dataset, batch_size=5, shuffle=True)
print(folder_path)

for batch in tqdm(train_loader, desc="Batch", leave=False):
    print(batch[0].shape)

print("Memory usage after loading data:", memory_usage(), "MB")

Memory usage before loading data: 548.43359375 MB
./data_o_e/SM4_P/dataset_v1


torch.Size([30000, 8])
torch.Size([8000, 8])
Memory usage after loading data: 549.16015625 MB


In [18]:
val_dataset = HDF5DatasetStatic(folder_path,'val', step=1, shuffle=True)
x_val, y_val = val_dataset.x_data, val_dataset.y_data

device(type='cuda', index=0)

In [34]:


start_time = time.time()
for i in range(100):
    for (x_batch, y_batch), (x_col_batch), (x_col_ic_batch, y_col_ic_batch) in tqdm(
        zip(train_loader, col_loader, init_loader), desc="Batch", leave=False
    ):
        pass

print("Time taken for 100 iterations:", time.time() - start_time, "seconds")

Time taken for 100 iterations: 70.26828598976135 seconds


In [40]:
import os
import psutil
import numpy as np
from torch.utils.data import DataLoader



# Print the number of samples in the dataset
#clean gc
import gc
gc.collect()

def memory_usage():
    """Returns the current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB

# Print memory usage before loading the data
print("Memory usage before loading data:", memory_usage(), "MB")

# Assuming HDF5Dataset is defined elsewhere
train_dataset0 = HDF5Dataset_static(folder_path, 'train', shuffle=True)
print("Memory usage after loading data:", memory_usage(), "MB")


train_loader0 = DataLoader(train_dataset0, batch_size=1, shuffle=True)

# Print memory usage after loading the data
print("Memory usage after loading data:", memory_usage(), "MB")



Memory usage before loading data: 390.33203125 MB
['./data_o_e/SM4_P/dataset_v1\\train\\train_data1.h5', './data_o_e/SM4_P/dataset_v1\\train\\train_data2.h5', './data_o_e/SM4_P/dataset_v1\\train\\train_data_init1.h5']
Memory usage after loading data: 390.34765625 MB
Memory usage after loading data: 390.4140625 MB


In [61]:
#assign train_dataset0 to x_train, y_train

x_train, y_train = train_dataset01[0]

x_train.shape, y_train.shape

(torch.Size([8]), torch.Size([4]))

In [36]:
# print how much ram it uses: train_dataset0

import os
import psutil
import numpy as np

# Print the number of samples in the dataset
#clean gc
import gc
gc.collect()

def memory_usage():
    """Returns the current process memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB

# Print memory usage before loading the data
print("Memory usage before loading data:", memory_usage(), "MB")

# Assuming HDF5Dataset is defined elsewhere
train_dataset01 = HDF5Dataset(folder_path, 'train', shuffle=False)
data_loader = DataLoader(train_dataset01, batch_size=3200, shuffle=False)
print("Memory usage after loading data:", memory_usage(), "MB")


Memory usage before loading data: 711.15234375 MB
Memory usage after loading data: 711.19921875 MB


In [41]:
x_train, y_train = next(iter(data_loader))

#print the number of batches in data_loader

len(data_loader)

12

In [14]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


# i just want the data to be on the gpu
for batch in tqdm(train_loader, desc="Batch", leave=False):
    x_train = batch[0].to(device)
    y_train = batch[1].to(device)
    break

x_train.requires_grad = True
y_train.requires_grad = True



cuda


torch.Size([24024, 8])

Time for create_col_points:  0.233933687210083
Time for create_col_points2:  0.04965686798095703
